## 📊 SVOMPTR Dataset Analysis & Cleaning Pipeline
This notebook analyzes the generated synthetic dataset for syntactic diversity (Simple, Compound, Complex), domain distribution (Technical vs Daily), and filters out low-quality translations using Multi-lingual Semantic Similarity.

In [ ]:
%%capture
!pip install datasets sentence-transformers spacy matplotlib seaborn
!python -m spacy download en_core_web_sm

### 1. Load Dataset

In [ ]:
import json
import os
import pandas as pd
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util

DATASET_PATH = "/content/drive/MyDrive/svomptr_auto_train/datasets/synthetic_100k_high_quality.jsonl"
CLEANED_PATH = "/content/drive/MyDrive/svomptr_auto_train/datasets/synthetic_100k_cleaned.jsonl"

# For local fast-testing, we check local colab storage first or fallback to local path
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = "./synthetic_100k_high_quality.jsonl" # Fallback

data = []
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, 'r') as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except:
                pass
df = pd.DataFrame(data)
print(f"✅ Loaded {len(df)} samples from {DATASET_PATH}.")

### 2. Sentence Complexity Analysis (Simple, Compound, Complex)
Using spaCy's dependency parser to check clause count and conjunctions.

In [ ]:
nlp = spacy.load("en_core_web_sm")

def classify_sentence(text):
    doc = nlp(text)
    num_verbs = sum(1 for token in doc if token.pos_ == "VERB")
    num_conjs = sum(1 for token in doc if token.pos_ in ["CCONJ", "SCONJ"])
    
    if num_verbs <= 1 and num_conjs == 0:
        return "Simple"
    elif num_conjs >= 1 and any(token.dep_ == "mark" or token.dep_ == "advcl" for token in doc):
        return "Complex"
    elif num_conjs >= 1:
        return "Compound"
    else:
        return "Simple"

# Sample 5000 rows for fast analysis
sample_df = df.sample(min(5000, len(df))) if len(df) > 0 else df
if len(sample_df) > 0:
    sample_df['complexity'] = sample_df['en'].apply(classify_sentence)
    
    plt.figure(figsize=(8,5))
    sns.countplot(x='complexity', data=sample_df, palette="viridis")
    plt.title("Distribution of Sentence Complexity")
    plt.xlabel("Sentence Type")
    plt.ylabel("Count")
    plt.show()
    
    print("Complexity Ratio:")
    print(sample_df['complexity'].value_counts(normalize=True) * 100)

### 3. Domain Analysis (Technical vs Daily Conversation)

In [ ]:
tech_keywords = ['data', 'system', 'research', 'analysis', 'medical', 'technology', 'network', 'algorithm', 'clinical', 'science', 'device', 'software', 'cloud', 'security', 'quantum']

def classify_domain(text):
    text_lower = text.lower()
    if any(kw in text_lower for kw in tech_keywords):
        return "Technical/Scientific"
    return "Daily/General"

if len(sample_df) > 0:
    sample_df['domain'] = sample_df['en'].apply(classify_domain)
    
    plt.figure(figsize=(6,4))
    sns.countplot(x='domain', data=sample_df, palette="magma")
    plt.title("Domain Distribution")
    plt.show()
    
    print("Domain Ratio:")
    print(sample_df['domain'].value_counts(normalize=True) * 100)

### 4. Semantic Similarity & Quality Filtering (LaBSE)
We use Language-agnostic BERT Sentence Embedding (LaBSE) which natively supports both English and Myanmar to compute cosine similarity. If the meaning deviates heavily (e.g., Score < 0.55), we drop the record to prevent "Data Poisoning".

In [ ]:
print("Loading Multi-lingual Embedding Model (LaBSE)... (This is 1.8GB, may take a moment)")
try:
    # LaBSE is state-of-the-art for mapping 109 languages into shared latent space
    embedder = SentenceTransformer('sentence-transformers/LaBSE')
except Exception as e:
    embedder = None
    print(f"Could not load LaBSE: {e}")

def filter_dataset(df_input, threshold=0.55):
    if embedder is None:
        return df_input
    
    print(f"🚀 Processing Semantic Similarity Scoring for {len(df_input)} rows...")
    en_embeddings = embedder.encode(df_input['en'].tolist(), batch_size=64, show_progress_bar=True, convert_to_tensor=True)
    my_embeddings = embedder.encode(df_input['my'].tolist(), batch_size=64, show_progress_bar=True, convert_to_tensor=True)
    
    cosine_scores = util.cos_sim(en_embeddings, my_embeddings).diagonal()
    df_input['similarity_score'] = cosine_scores.cpu().numpy()
    
    # Filter out bad translations
    cleaned_df = df_input[df_input['similarity_score'] >= threshold]
    
    bad_examples = df_input[df_input['similarity_score'] < threshold].head(5)
    
    print("\n" + "="*40)
    print(f"📊 Original size  : {len(df_input)}")
    print(f"🧼 Cleaned size   : {len(cleaned_df)}")
    print(f"🗑️ Removed records : {len(df_input) - len(cleaned_df)}")
    
    if len(bad_examples) > 0:
        print("\n⚠️ Example of removed bad translations:")
        for idx, row in bad_examples.iterrows():
            print(f"Similarity: {row['similarity_score']:.2f} | EN: {row['en']} -> MY: {row['my']}")
            
    return cleaned_df

# Apply on full dataset if available
if len(df) > 0:
    cleaned_data = filter_dataset(df, threshold=0.55)
    
    # Save clean dataset
    export_dir = os.path.dirname(CLEANED_PATH)
    if not os.path.exists(export_dir): os.makedirs(export_dir, exist_ok=True)
    
    with open(CLEANED_PATH, 'w', encoding='utf-8') as f:
        # Drop the score before saving for training
        for record in cleaned_data.drop(columns=['similarity_score'], errors='ignore').to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            
    print(f"\n✅ Fully cleaned dataset saved to: {CLEANED_PATH}")
    print("🎯 You should now update 'SVOMPTR_9B_AutoTrain_Unsloth.ipynb' to use this cleaned jsonl file!")